# 第 5 章数据完整性 —— 全仓污染面枚举的销号（Colab / Drive 侧）

**只做一件事**：把 `docs/data_integrity_open_items.md` 里唯一还开着的那条——「污染面枚举**全仓**
未封闭」——跑到能销号。第 5 章用到的那批数据集**章内已封闭**；这里封的是仓级。

## 判据：三个零

| 信号 | 通过值 |
|---|---|
| `[ENUM MISS]` 行 | 0 条 |
| `[SHADOWED ]` 行 | 0 条 |
| 进程退出码 | `0` |

三者同时成立才算销号。**「重跑一次数目对上了」不算数**——2026-08-16 首轮 range pass 报
`datasets: 24`（漏掉两个相邻的含噪集），次轮同一份代码报 `26`；**次轮本身就是那次重跑**。
所以判据是「账本逐名 `stat` 反查枚举」，不是枚举自己跟自己对得上。

`OVERLAP` 行**不是失败**，是审计的**发现**（哪个数据集与哪个评估集共种子），退出码不受它影响。

## 为什么代码不从 Drive 取

Drive 上的 `rl_v2_5/` 是**同步副本，不是 clone**。它自带的 `scripts/audit_seed_overlap.py`
停在最后一次同步的版本，很可能早于这三个提交：

| 提交 | 改了什么 | 缺了它会怎样 |
|---|---|---|
| `0aa42ac` | 单层 glob → `rglob` | `fql_succession/` 下嵌套一层的数据集整批不进枚举 |
| `8d03fcc` | 账本对帐 ＋ 退出码 `2` | **没有对帐块可判**，上面三条判据全部落空 |
| `95a8302` | `--data-dir` / `--benchmarks-dir` | 无法「代码从 git 取、数据从 Drive 读」 |
| `06e6d1f` | 枚举跟随符号链接 | Drive 侧**三个**数据集整个掉出枚举 —— 2026-08-21 那轮就是这么栽的 |

跑旧脚本会**静默重现旧行为**，且输出看着一切正常。所以：**代码 `git clone`，数据 `--data-dir`
指 Drive**，两棵树本来就不必是同一个。

> 本 notebook **取代** `ch5_data_integrity_probe.ipynb` 里「§2 探针 B」那几格就地
> `!python -m scripts.audit_seed_overlap`（在 Drive 树里跑 Drive 自带的脚本，正是上面说的坑）。
> 该 notebook 的 §1 探针 A 与 §3 ①-c 补评**已闭环，不要重跑**。

**运行时**：CPU 即可，**不要开 GPU**（纯 I/O 与 JSON 解析，几分钟，省配额）。

## 0. 挂载 Drive，代码从 git 取

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# --depth 1 足够：这里只跑脚本，不查历史。
# 已有 clone 就拉到最新 —— 同一个会话里重跑而不刷新，等于继续跑旧代码，而本 notebook 的判据
# 恰恰依赖脚本版本。reset --hard 会丢掉上一轮 --record 改的账本（第 6 节导出过就没关系）。
!if [ -d /content/rl_v2/.git ]; then git -C /content/rl_v2 fetch --depth 1 origin main && git -C /content/rl_v2 reset --hard FETCH_HEAD; else git clone --depth 1 https://github.com/davidjin1214/auvnav-rl_v2.git /content/rl_v2; fi
%cd /content/rl_v2
!git log --oneline -1

## 1. 环境自检 —— 这三条不过就别往下跑

第 (2) 条一箭双雕：`--help` 能跑通说明 `import` 链走完了（依赖齐），输出里有 `--data-dir`
说明这份 clone 新过 `95a8302`。

In [ ]:
import os, re, subprocess, sys
from pathlib import Path

DRIVE = Path('/content/drive/MyDrive/Colab Notebooks/new_offRL/rl_v2_5')
DATA  = DRIVE / 'offline_data'
BENCH = DRIVE / 'benchmarks'


def run(cmd):
    """Stream a subprocess into the cell output and hand back (exit code, text).

    Deliberately not `!python ...`, which the rest of this repo's notebooks use: a non-zero exit
    from a `!` line does not fail the cell, and here the exit code *is* the verdict (2 = the
    enumeration is incomplete). The list form also sidesteps quoting the Drive path, which
    contains a space. Colab drops inherited stderr, so stderr is folded into stdout.
    """
    print('$', ' '.join(str(c) for c in cmd))
    proc = subprocess.Popen([str(c) for c in cmd], stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = []
    for line in proc.stdout:
        print(line, end='')
        lines.append(line)
    rc = proc.wait()
    print(f'--- exit={rc} ---')
    return rc, ''.join(lines)


# (1) Drive 两棵树都在
assert DATA.is_dir(),  f'找不到 {DATA} —— 确认 Drive 路径拼写（含空格）'
assert BENCH.is_dir(), f'找不到 {BENCH}'

# (2) clone 的脚本够新，且依赖装齐
_h = subprocess.run([sys.executable, '-m', 'scripts.audit_seed_overlap', '--help'],
                    capture_output=True, text=True)
assert _h.returncode == 0, f'--help 没跑通（多半缺 gymnasium）：\n{_h.stdout}\n{_h.stderr}'
assert '--data-dir' in _h.stdout, 'clone 里的脚本没有 --data-dir —— 这份 clone 太旧，别继续'
print('OK: 脚本带 --data-dir，依赖齐')

# (3) 规模。这两个数是判读的背景，不是判据 —— 判据在第 5 节
n_top = len([e for e in os.scandir(DATA) if e.is_dir()])
n_led = len([l for l in Path('scripts/offline_dataset_ledger.txt').read_text(encoding='utf-8').splitlines()
             if l.split('#', 1)[0].strip()])
print(f'Drive offline_data/ 顶层目录: {n_top}    clone 账本里的名字: {n_led}')

# (4) 顶层条目的形态。2026-08-21 那轮有三个数据集 os.scandir 列得到、metadata.json stat 得到，
# 却整个掉出枚举 —— 那时的 rglob 不进符号链接目录。枚举已改成照样往下走，这里只是把形态摆出来，
# 好在下面万一还报 [ENUM MISS] 时，能一眼分清「是链接」还是「is_dir() 在 FUSE 上不老实」。
odd = [e for e in sorted(os.scandir(DATA), key=lambda e: e.name)
       if e.is_symlink() or not e.is_dir()]
for e in odd:
    print(f'  {e.name}: is_symlink={e.is_symlink()} is_dir={e.is_dir()} '
          f'metadata={(DATA / e.name / "metadata.json").is_file()}')
print(f'  形态异常的顶层条目: {len(odd)}' if odd else '  顶层无符号链接，也无 is_dir() 为假的条目')

## 2. 用哪些 manifest —— 两棵树先对一遍

`benchmarks/` 是**被 git 跟踪**的，所以 clone 里那份是权威副本；Drive 那份是同步来的，可能**少**了
新提交的，也可能**多**出在 Colab 上就地生成、从未回传进 git 的——后者事先不知道叫什么，只能打出来看。

覆盖面缺一边都不算仓级封闭，所以下面取**并集**：以 Drive 那份为底，把 clone 独有的补进去。
差集本身就是读数——先打印出来。

In [ ]:
import json, shutil


def manifest_names(root: Path) -> set:
    """Names of every JSON under root that actually is a benchmark manifest."""
    names = set()
    for p in sorted(root.rglob('*.json')):
        try:
            if 'episodes' in json.loads(p.read_text(encoding='utf-8')):
                names.add(str(p.relative_to(root)).replace('\\', '/'))
        except Exception as exc:
            print(f'  [跳过] {p} — {exc}')
    return names


git_m, drive_m = manifest_names(Path('benchmarks')), manifest_names(BENCH)
print(f'clone benchmarks/: {len(git_m)}    Drive benchmarks/: {len(drive_m)}\n')
for n in sorted(git_m - drive_m):
    print(f'  [仅 clone] {n}')
for n in sorted(drive_m - git_m):
    print(f'  [仅 Drive] {n}   ← 就地生成、没进 git')
if not (git_m ^ drive_m):
    print('  两棵树的 manifest 完全一致')

# 并集：Drive 打底 + clone 独有的补进来。只是拷 JSON，不动任何一棵原树。
BENCH_USE = BENCH
if git_m - drive_m:
    BENCH_USE = Path('/content/benchmarks_union')
    if BENCH_USE.exists():
        shutil.rmtree(BENCH_USE)
    shutil.copytree(BENCH, BENCH_USE)
    for n in sorted(git_m - drive_m):
        dst = BENCH_USE / n
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(Path('benchmarks') / n, dst)
    print(f'\n→ 用并集 {BENCH_USE}（{len(manifest_names(BENCH_USE))} 个 manifest）')
else:
    print(f'\n→ 直接用 {BENCH_USE}')

## 3. 第一遍：`--record` 把名字收进账本

账本是**跨机并集，只增不删**——短列举不会让它变瘦。这一遍也会打印对帐块，但**不以它为准**：
它跑的时候，Drive 独有的名字还没进账本，反查无从谈起。

⚠ `--record` 只能记下**枚举看得见**的名字，而枚举本身正是嫌疑对象——所以一次短列举会少记，
这**不构成销号**，只是让账本长期变厚。

In [ ]:
AUDIT = [sys.executable, '-m', 'scripts.audit_seed_overlap',
         '--data-dir', DATA, '--benchmarks-dir', BENCH_USE]

rc_record, out_record = run(AUDIT + ['--record'])

## 4. 第二遍：判对帐

同一份代码、同一棵数据树，但账本已含第一遍收到的名字。**这一遍的退出码才是判据。**

In [ ]:
rc_audit, out_audit = run(AUDIT)

## 5. 判定

In [ ]:
lines    = out_audit.splitlines()
miss     = [l for l in lines if '[ENUM MISS]' in l]
shadowed = [l for l in lines if '[SHADOWED' in l]
unrec    = [l for l in lines if '[new' in l]
scale    = re.search(r'^datasets: (\d+)\s+manifests: (\d+)', out_audit, re.M)

# OVERLAP 明细：dataset 在 [OVERLAP] 那行，manifest 在其下的缩进行
pairs, cur = [], None
for l in lines:
    m1 = re.match(r'\[(?:OVERLAP|clean)\s*\]\s+(\S+)', l)
    if m1:
        cur = m1.group(1)
    m2 = re.match(r'\s+OVERLAP \d+/\d+ with (\S+)', l)
    if m2 and cur:
        pairs.append((cur, m2.group(1)))

print('=' * 78)
print(f'[ENUM MISS] {len(miss):>3d}      [SHADOWED ] {len(shadowed):>3d}      exit={rc_audit}')
print('=' * 78)
if rc_audit == 0 and not miss and not shadowed:
    print('✅ 三个零成立 —— 本次枚举覆盖了账本里每一个名字，仓级污染面可销号。')
    print('   抄回 docs/data_integrity_open_items.md 销号条件下方时，连同下面这三样一起写：')
    print('     · 日期与 clone 的 commit（上面 git log 那行）')
    print(f'     · 规模：{scale.group(0) if scale else "（未解析到 datasets/manifests 行）"}')
    print(f'     · OVERLAP {len(pairs)} 处（明细见下）')
else:
    print('❌ 未通过。枚举不完整，任何「只有 N 个数据集受影响」的结论在解决它之前都无依据。')
    for l in miss + shadowed:
        print('   ', l.strip())
    print('   下一步：ENUM MISS = 该名字 stat 得到但列举没返回。先看第 1 节第 (4) 步的形态普查：')
    print('           两遍都缺、且形态异常 → 结构性，是代码问题；只缺一遍 → 瞬时短列举，重跑。')
    print('           SHADOWED = scandir 看得到而枚举没返回；枚举已跟随符号链接，仍报说明下行逻辑漏了它。')
print()
if unrec:
    print(f'[new] {len(unrec)} 个名字还没进账本 —— 第 3 节的 --record 应已收下，若仍在说明两遍之间列举变了：')
    for l in unrec:
        print('   ', l.strip())
    print()
print(f'OVERLAP 明细（{len(pairs)} 处，**不影响退出码**——这是审计的发现，不是失败）：')
for ds, mf in pairs:
    print(f'  {ds}\n      × {mf}')
if pairs:
    print('\n要把某一对从「种子区间相交」升级为「逐条同一实例」，把它填进第 7 节的 VERIFY_PAIR：')
    print(f'  VERIFY_PAIR = {pairs[0]!r}')
print('\n提醒：数目自洽（datasets 数跟上次一样）**不能**用来销号 —— 见顶部说明。')

## 6. 回传账本

被 `--record` 改的是 **clone 里**的 `scripts/offline_dataset_ledger.txt`，Colab 一断开就没了。
不带回本机提交，这次收到的名字下轮就不存在，下次列举又没有东西反查它。

In [ ]:
!git diff --stat -- scripts/offline_dataset_ledger.txt
!git diff -- scripts/offline_dataset_ledger.txt | head -60

In [ ]:
LEDGER      = Path('scripts/offline_dataset_ledger.txt')
LEDGER_BACK = DRIVE.parent / 'offline_dataset_ledger.from_colab.txt'   # 落在项目副本之外

if subprocess.run(['git', 'diff', '--quiet', '--', str(LEDGER)]).returncode:
    shutil.copy(LEDGER, LEDGER_BACK)      # 落 Drive 打底：浏览器可能拦下载
    print(f'已写 {LEDGER_BACK}')
    try:
        from google.colab import files
        files.download(str(LEDGER))
    except Exception as exc:
        print(f'（files.download 不可用：{exc} —— 用上面那份 Drive 副本）')
    print('\n本机应用：覆盖 scripts/offline_dataset_ledger.txt 后')
    print('  git diff --stat scripts/offline_dataset_ledger.txt   # 只应有新增行；有删除行就停下')
    print('  git add scripts/offline_dataset_ledger.txt && git commit')
else:
    print('账本无改动 —— Drive 侧没有账本以外的数据集名。')

## 7. 可选：实例级钉死（`--verify`）

range pass 只证明「种子区间相交」。`--verify` 重放 reset 的 RNG，把相交的那些种子逐条与冻结的
manifest 条目比对（`flow_time` / `start_xy` / `goal_xy` / `initial_heading`），**不跑仿真**，把
「相交」升级为「逐条同一实例」。

它要按 `metadata.json` 里的 `flow_path`（仓库相对路径）打开流场，而 clone 里没有 `wake_data/`
——所以先把 Drive 那棵软链进来。**只软链 `wake_data`**：`offline_data` 一律走 `--data-dir`。
枚举自 `06e6d1f` 起会跟随符号链接（在此之前，链过去正好触发 `[SHADOWED]` 那类失效），但没必要
多绕一层间接 —— `--data-dir` 就是为这件事加的。

In [ ]:
RUN_VERIFY  = False        # 第 5 节报了你想钉死的 OVERLAP 时才打开
VERIFY_PAIR = ('', '')     # 照抄第 5 节打印的那一行

if RUN_VERIFY:
    link = Path('wake_data')
    if not link.exists():
        link.symlink_to(DRIVE / 'wake_data')
    rc_v, _ = run(AUDIT + ['--verify', VERIFY_PAIR[0], VERIFY_PAIR[1]])
    print('判读：identical == checked 即逐条相同；checked 为 0 说明这一对本来就不相交。')
else:
    print('RUN_VERIFY = False —— 跳过。')

## 8. 把 Drive 独有的 manifest 打包带回

第 2 节已经算出是哪些：`drive_m - git_m`。它们是**实际用过的评估集**，而 `benchmarks/` 是被 git
跟踪的目录——本意就是让评估集冻得住、可复现。这些没进 git，于是**任何只从 clone 出发的审计都
覆盖不到它们**，上面那个封闭也就不可复现。

不在本机重生成：`auv_nav/env.py` 在 benchmark 协议冻结（`9b96a7d`）之后被 `813096e` 动过，
重跑生成器不保证逐条复现；而这是评估记录，要的是**原字节，不是等价物**。

`arcname` 用的就是它在 `benchmarks/` 下的相对路径，所以本机把 zip 解到 `benchmarks/` 即原样归位。

In [ ]:
import zipfile

extra = sorted(drive_m - git_m)
if extra:
    bundle = Path('/content/drive_only_manifests.zip')
    with zipfile.ZipFile(bundle, 'w', zipfile.ZIP_DEFLATED) as z:
        for n in extra:
            z.write(BENCH / n, arcname=n)
    print(f'{bundle}  {bundle.stat().st_size / 1024:.0f} KB  {len(extra)} 个文件')
    for n in extra:
        print(f'  {(BENCH / n).stat().st_size / 1024:6.1f} KB  {n}')
    from google.colab import files
    files.download(str(bundle))
    print('\n本机：unzip -o drive_only_manifests.zip -d benchmarks/ 然后 git add benchmarks/')
else:
    print('Drive 没有 git 之外的 manifest —— 无需回传。')